# Space-Grade Mechanical Fault Detector — LibreLane RTL-to-GDSII (GF180)

End-to-end **single-macro** Classic flow inside the `hpretl/iic-osic-tools:chipathon26`
Docker container, targeting the **GF180mcuD** PDK.

| Stage | Detail |
|---|---|
| Design | `top` — Interleaved Tri-Axis Goertzel (ITAG) vibration-fault detector |
| PDK | `gf180mcuD`  (wafer-space fork) |
| Standard cells | `gf180mcu_fd_sc_mcu7t5v0` |
| System clock | 16 MHz (62.5 ns) |
| SPI clock | 2 MHz (clk / 8 via `clk_divider`) |

**Stage split (mirrors ~/riscv-soc/librelane pattern):**
- Cell 9  — Synthesis through Detailed Routing (runtime ~15-30 min)
- Cell 11 — Signoff: GDS streamout, DRC, LVS, STA report

Run flags at the top of Cell 3 let you skip stages already completed.

## Step 0.1 — Configuration

Paths, container name, PDK identifiers, and run-stage flags.

In [ ]:
from pathlib import Path
import shutil, subprocess, textwrap, json, csv

# --- Container -------------------------------------------------------
CONTAINER_NAME  = "gf180"         # docker run --name gf180 ...

# --- Host paths -------------------------------------------------------
PROJECT_ROOT    = Path.home() / "Space-Grade-Mechanical-Fault-Detector"
NB_DIR          = PROJECT_ROOT / "librelane"
HOST_WORKSPACE  = Path.home() / "eda" / "designs" / "fault_detector"

# --- Container paths (bind-mounted) -----------------------------------
CONTAINER_WORKSPACE = "/foss/designs/fault_detector"
CONTAINER_PDK_ROOT  = "/foss/designs/chipathon_padring/template/gf180mcu"

# --- PDK identifiers --------------------------------------------------
PDK_NAME       = "gf180mcuD"
STD_CELL_LIB   = "gf180mcu_fd_sc_mcu7t5v0"

# --- Run-stage flags (flip True one-at-a-time as you progress) --------
RUN_STAGE_FILES = True   # Step 1 -- copy rtl/ tb/ librelane/ into bind-mount
RUN_COCOTB      = True   # Step 2 -- RTL cocotb verification
RUN_VERIFY_PDK  = True   # Step 3 -- check container + PDK
RUN_PLACE_ROUTE = True   # Step 4 -- synth through detail-routing
RUN_SIGNOFF     = True   # Step 5 -- GDS + DRC + LVS + STA

print("Configuration loaded.")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Host ws      : {HOST_WORKSPACE}")
print(f"  Container    : {CONTAINER_NAME}  ->  {CONTAINER_WORKSPACE}")

## Step 0.2 — Helpers (`run_or_print`, `ok`)

`run_or_print` prints every command, then executes it inside `docker exec gf180 bash -lc ...` only when the matching `RUN_*` flag is `True`.

In [ ]:
def ok(label, cond, detail=""):
    tag = "OK " if cond else "!! "
    print(f"{tag}{label}" + (f"  -- {detail}" if detail else ""))
    return cond


def run_or_print(cmd, do_it, *, shell_on_container=False, timeout=None):
    """Print command; execute inside container when do_it=True.

    If shell_on_container=True, cmd is a multi-line bash script run via
    docker exec ... bash -lc.  Otherwise cmd is an argv list on the host.
    Returns the CompletedProcess when executed, None otherwise.
    """
    if shell_on_container:
        print(f"$ docker exec {CONTAINER_NAME} bash -lc '<script>'")
        print(textwrap.indent(cmd, "  | "))
    else:
        print("$ " + " ".join(str(x) for x in cmd))
    if not do_it:
        print("  (skipped -- flip the RUN_* flag to execute)\n")
        return None
    if shell_on_container:
        result = subprocess.run(
            ["docker", "exec", CONTAINER_NAME, "bash", "-lc", cmd],
            timeout=timeout,
        )
    else:
        result = subprocess.run(cmd, timeout=timeout)
    result.check_returncode()
    print()
    return result

## Step 1 — Stage project files into the bind-mount

Copies `rtl/`, `tb/`, and `librelane/` from the project repo into
`~/eda/designs/fault_detector/` (container: `/foss/designs/fault_detector/`).
Idempotent: removes and re-stages on every run.

In [ ]:
if RUN_STAGE_FILES:
    if HOST_WORKSPACE.exists():
        shutil.rmtree(HOST_WORKSPACE)
    HOST_WORKSPACE.mkdir(parents=True)
    for sub in ("rtl", "tb", "librelane"):
        src = PROJECT_ROOT / sub
        dst = HOST_WORKSPACE / sub
        shutil.copytree(src, dst)
        n = sum(1 for _ in dst.rglob("*") if _.is_file())
        print(f"  {sub}/  -- {n} file(s)")
    (HOST_WORKSPACE / "build").mkdir(exist_ok=True)
    print(f"\nStaged at {HOST_WORKSPACE}")
else:
    print("(dry-run) would stage:")
    for sub in ("rtl", "tb", "librelane"):
        print(f"  {PROJECT_ROOT / sub}  ->  {HOST_WORKSPACE / sub}")

ok("Host workspace present", HOST_WORKSPACE.exists(), str(HOST_WORKSPACE))

## Step 2 — RTL Verification (cocotb)

Runs the cocotb testbench suite inside the container:

- `test-spi`      — SPI boot-write sequence + burst-read framing
- `test-goertzel` — ITAG Goertzel Q8.15 arithmetic, 3-axis update, `sample_done` cadence
- `test-top`      — Full chip: single-axis faults (X/Y/Z), simultaneous 3-axis excitation,
                    fault-clear, no-fault baseline, ITAG structural invariants

Both Icarus Verilog and cocotb ship pre-installed in `hpretl/iic-osic-tools:chipathon26`.

In [ ]:
cocotb_script = textwrap.dedent(f"""
    set -e
    cd {CONTAINER_WORKSPACE}/tb
    make clean
    make test-spi
    make test-goertzel
    make test-top
""").strip()

run_or_print(cocotb_script, RUN_COCOTB, shell_on_container=True, timeout=600)

## Step 3 — Verify container + GF180 PDK

Confirms:
1. The `gf180` container is running (`docker ps`).
2. The GF180 PDK is available at the expected path inside the container.

In [ ]:
if RUN_VERIFY_PDK:
    proc = subprocess.run(
        ["docker", "ps", "--filter", f"name={CONTAINER_NAME}", "--format", "{{.Names}}"],
        capture_output=True, text=True,
    )
    container_up = CONTAINER_NAME in proc.stdout
    ok(f"Container '{CONTAINER_NAME}' running", container_up)

    if container_up:
        pdk_check = subprocess.run(
            ["docker", "exec", CONTAINER_NAME, "test", "-d",
             f"{CONTAINER_PDK_ROOT}/{PDK_NAME}"],
            capture_output=True,
        )
        ok(f"GF180 PDK present at {CONTAINER_PDK_ROOT}/{PDK_NAME}",
           pdk_check.returncode == 0)
    else:
        print("!! Skipping PDK check -- container not up.")
        print("   Run:  docker start gf180")
else:
    print("(skipped) RUN_VERIFY_PDK = False")

## Step 4 — Synthesis through Detailed Routing

Runs the LibreLane Classic flow from Synthesis through Detailed Routing on
`librelane/top.yaml`.  Outputs land in `build/top/`.

**Key parameters (set in `librelane/top.yaml`):**

| Parameter | Value | Rationale |
|---|---|---|
| `CLOCK_PERIOD` | 62.5 ns | 16 MHz system clock |
| `SYNTH_STRATEGY` | `AREA 0` | compact Goertzel v1/v2 register chains |
| `FP_CORE_UTIL` | 40 % | comfortable starting utilisation on GF180 |
| `PL_TARGET_DENSITY_PCT` | 55 % | avoid routing congestion from dense TMR regs |

Runtime: ~15-30 min.  Stops at `OpenROAD.DetailedRouting` so Step 5 can
be restarted independently.

In [ ]:
place_route = textwrap.dedent(f"""
    set -e
    cd {CONTAINER_WORKSPACE}
    librelane librelane/top.yaml \\
        --pdk {PDK_NAME} \\
        --pdk-root {CONTAINER_PDK_ROOT} \\
        --scl {STD_CELL_LIB} \\
        --save-views-to {CONTAINER_WORKSPACE}/build/top \\
        --to OpenROAD.DetailedRouting
""").strip()

run_or_print(place_route, RUN_PLACE_ROUTE, shell_on_container=True, timeout=3600)

## Step 5 — Signoff (GDS streamout, DRC, LVS, STA)

Resumes from the Detailed Routing state and runs:
- Routing-obstruction removal, antenna check, fill insertion
- RCX parasitic extraction
- Multi-corner post-PnR STA
- Magic + KLayout GDS streamout
- DRC (Magic) + LVS (Netgen)

Outputs: `build/top/final/gds/top.gds`, `build/top/final/lef/top.lef`,
`build/top/nl/top.nl.v`, `build/top/metrics.csv`.

In [ ]:
signoff = textwrap.dedent(f"""
    set -e
    cd {CONTAINER_WORKSPACE}
    librelane librelane/top.yaml \\
        --pdk {PDK_NAME} \\
        --pdk-root {CONTAINER_PDK_ROOT} \\
        --scl {STD_CELL_LIB} \\
        --save-views-to {CONTAINER_WORKSPACE}/build/top \\
        --from Odb.RemoveRoutingObstructions
""").strip()

run_or_print(signoff, RUN_SIGNOFF, shell_on_container=True, timeout=3600)

## Step 6 — Signoff Metrics Summary

Parses `build/top/metrics.csv` and prints:
- Die area, setup/hold worst slack
- DRC / LVS / antenna violation counts
- Total power estimate

Prints **CLEAN** or **VIOLATIONS PRESENT** verdict.

In [ ]:
metrics_path = HOST_WORKSPACE / "build" / "top" / "metrics.csv"

WANTED = [
    ("design__die__bbox",           "Die bbox (um)"),
    ("design__core__bbox",          "Core bbox (um)"),
    ("timing__setup__ws",           "Setup WS (ns)"),
    ("timing__hold__ws",            "Hold WS (ns)"),
    ("magic__drc_errors",           "Magic DRC errors"),
    ("design__lvs_errors",          "LVS errors"),
    ("antenna__violating__nets",    "Antenna violating nets"),
    ("power__total",                "Total power (W)"),
    ("design__instance__count",     "Cell count"),
]

if not metrics_path.exists():
    print(f"!! metrics.csv not found at {metrics_path}")
    print("   Run Step 5 (RUN_SIGNOFF=True) first.")
else:
    with open(metrics_path) as f:
        rows = list(csv.DictReader(f))
    m = rows[-1] if rows else {}

    print(f"{'Metric':<35}  {'Value'}")
    print("-" * 55)
    violations = False
    for key, label in WANTED:
        val = m.get(key, "N/A")
        flag = ""
        if key in ("magic__drc_errors", "design__lvs_errors") and val not in ("0", "N/A"):
            flag = "  WARNING"
            violations = True
        if key in ("timing__setup__ws", "timing__hold__ws"):
            try:
                if float(val) < 0:
                    flag = "  WARNING"
                    violations = True
            except ValueError:
                pass
        print(f"  {label:<33}  {val}{flag}")

    print()
    if violations:
        print("!! VIOLATIONS PRESENT -- review flags above before tapeout.")
    else:
        print("OK CLEAN -- all metrics within bounds.")